This is jupyter notebook for teamates who are not familar with command line interface

In [2]:
import pandas as pd

In [5]:
path = '../data/arcgis_img.csv'
df= pd.read_csv(path)
df.to_csv('../data/arcgis_img.tsv',sep='\t',index=False)

In [4]:
path = '../data/haunted_places.tsv'
df= pd.read_csv(path, sep='\t')

In [15]:
# The number of unique cities
print(len(df.city.unique()))

4386


In [33]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime

##################################################################################3
# Source 1 : https://www.timeanddate.com/astronomy/usa

def webcrawl_daylight():
    # URL of interest
    url = "https://www.timeanddate.com/astronomy/usa"
    
    # 1. Fetch the page content
    response = requests.get(url)
    if response.status_code != 200:
        print(f"Failed to retrieve page, status code: {response.status_code}")
        return None, None

    # 2. Parse the HTML
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # 3. Find the target table (class="zebra fw tb-sm zebra")
    table = soup.find('table', {'class': 'zebra fw tb-sm zebra'})
    if not table:
        print("Could not find the expected table on the page.")
        return None, None

    data = []

    # 4. Loop over each row in the table (skipping the header)
    rows = table.find_all('tr')[1:]  # first row is the header
    for row in rows:
        cols = row.find_all('td')
        if len(cols) < 3:
            # Not a valid data row
            continue

        # Column 0: City/State info (e.g. "Adak (AK)")
        city_state_full = cols[0].get_text(strip=True)

        # Attempt to parse city and state. 
        # The format might be something like "Adak (AK)" or "Akron – Ohio – USA"
        # Adjust parsing logic according to actual structure you see on the site.
        # For demonstration, let's do a simple split:
        if " – " in city_state_full:
            parts = [p.strip() for p in city_state_full.split("–")]
            if len(parts) >= 2:
                city = parts[0]
                state = parts[1]
            else:
                city = city_state_full
                state = "Unknown"
        else:
            # If the string is like "Adak (AK)"
            city = city_state_full
            # Extract something in parentheses as state, e.g. (AK)
            # This is optional and depends on the site’s structure
            if "(" in city_state_full and ")" in city_state_full:
                state = city_state_full[city_state_full.index("(")+1 : city_state_full.index(")")]
                # Also remove the parentheses from city name
                city = city_state_full.split("(")[0].strip()
            else:
                state = "Unknown"

        # Column 1: Sunrise time (might include '↑')
        sunrise_str = cols[1].get_text(strip=True)
        # Column 2: Sunset time (might include '↓')
        sunset_str = cols[2].get_text(strip=True)

        # Remove unwanted symbols/characters
        for arrow in ['↑', '↓']:
            sunrise_str = sunrise_str.replace(arrow, '').strip()
            sunset_str = sunset_str.replace(arrow, '').strip()

        # 5. Parse times into datetime objects so we can compute the difference
        # Use a dummy date to parse times
        date_str = "2025-01-01"  
        
        try:
            sunrise_dt = datetime.strptime(date_str + " " + sunrise_str, "%Y-%m-%d %I:%M %p")
            sunset_dt = datetime.strptime(date_str + " " + sunset_str, "%Y-%m-%d %I:%M %p")
            # Compute difference in hours
            daylight_minutes = (sunset_dt - sunrise_dt).total_seconds() / 60
        except Exception as e:
            # If parsing fails, skip
            print(f"Parsing error for city={city}, sunrise={sunrise_str}, sunset={sunset_str}, error={e}")
            continue
        
        data.append({
            "city": city,
            "state": state,
            "sunrise": sunrise_dt.strftime("%I:%M %p"),
            "sunset": sunset_dt.strftime("%I:%M %p"),
            "daylight_minutes": daylight_minutes
        })

    # 6. Create a DataFrame
    df = pd.DataFrame(data)

    avg_major_cities = df['daylight_minutes'].mean()

    # 7. Calculate average daylight hours by State
    #avg_by_state = (
    #    df.groupby("state")["daylight_minutes"]
    #      .mean()
    #      .reset_index()
    #      .rename(columns={"daylight_minutes": "avg_daylight_minutes"})
    #)

    return df, avg_major_cities

def generate_daylight_avg_by_state():
    # avg_by_state has two columns: [state, avg_daylight_minutes]
    df, avg_daylight_of_major_cities = webcrawl_daylight()
    if df is not None and avg_daylight_of_major_cities is not None:
        print("\n--- Raw Data (City-Level) ---")
        print(df.head(20))

        print("\n--- Average Daylight Hours of major cities ---")
        #print(avg_daylight_of_major_cities.sort_values("avg_daylight_minutes", ascending=False))
    #avg_daylight_of_major_cities.to_csv('../data/daylight_s1')
    return round(avg_daylight_of_major_cities,1)

generate_daylight_avg_by_state()


--- Raw Data (City-Level) ---
           city state   sunrise    sunset  daylight_minutes
0          Adak    AK  08:21 AM  07:34 PM             673.0
1        Albany    NY  06:23 AM  05:49 PM             686.0
2   Albuquerque    NM  06:30 AM  06:06 PM             696.0
3          Ames    IA  06:42 AM  06:09 PM             687.0
4     Anchorage    AK  07:46 AM  06:36 PM             650.0
5     Annapolis    MD  06:32 AM  06:03 PM             691.0
6       Atlanta    GA  07:00 AM  06:37 PM             697.0
7       Augusta    ME  06:08 AM  05:32 PM             684.0
8        Austin    TX  06:52 AM  06:32 PM             700.0
9     Baltimore    MD  06:32 AM  06:03 PM             691.0
10  Baton Rouge    LA  06:25 AM  06:06 PM             701.0
11     Billings    MT  06:44 AM  06:06 PM             682.0
12     Bismarck    ND  07:14 AM  06:35 PM             681.0
13        Boise    ID  07:13 AM  06:39 PM             686.0
14       Boston    MA  06:12 AM  05:39 PM             687.0
15  Carso

686.2

In [34]:
temp = 5
x = {"A":[1,2,3,4],"B":[5,6,7,8]}
df = pd.DataFrame(x)
df["C"] = df["B"] - temp
df


,A,B,C
0,1,5,0
1,2,6,1
2,3,7,2
3,4,8,3


In [28]:
import cv2
import pytesseract
import numpy as np 
import re 

# If Tesseract is not in your PATH, uncomment and specify its location:
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

# Hard-coded dictionary of approximate center coordinates for each state:
states_coords = {
    "Alabama": (1480, 890),
    "Arizona": (560, 830),
    "Arkansas": (1270, 800),
    "California": (250, 695),
    "Colorado": (780, 605),
    "Connecticut": (2000, 485),
    "Delaware": (1850, 615),
    "Florida": (1660, 1080),
    "Georgia": (1615, 920),
    "Idaho": (450, 360),
    "Illinois": (1380, 550),
    "Indiana": (1490, 590),
    "Iowa": (1230, 460),
    "Kansas": (1040, 640),
    "Kentucky": (1560, 670),
    "Louisiana": (1280, 1000),
    "Maine": (2140, 310),
    "Maryland": (1850, 620),
    "Massachusetts": (1995, 440),
    "Michigan": (1560, 420),
    "Minnesota": (1200, 230),
    "Mississippi": (1380, 880),
    "Missouri": (1270, 630),
    "Montana": (630, 195),
    "Nebraska": (1000, 480),
    "Nevada": (360, 590),
    "New Hampshire": (2060, 390),
    "New Jersey": (1930, 550),
    "New Mexico": (760, 820),
    "New York": (1920, 410),
    "North Carolina": (1760, 760),
    "North Dakota": (990, 170),
    "Ohio": (1630, 545),
    "Oklahoma": (1080, 770),
    "Oregon": (225, 360),
    "Pennsylvania": (1800, 540),
    "Rhode Island": (2060, 485),
    "South Carolina": (1700, 850),
    "South Dakota": (980, 340),
    "Tennessee": (1510, 760),
    "Texas": (1000, 930),
    "Utah": (560, 590),
    "Vermont": (1980, 315),
    "Virginia": (1800, 680),
    "Washington": (235, 178),
    "West Virginia": (1710, 650),
    "Wisconsin": (1360, 330),
    "Wyoming": (715, 410),
}

# Path to your image (2256x1270):
image_path = "../data/img/mental_health_04.jpg"

# Read the image
image = cv2.imread(image_path)

crop_w, crop_h = 140, 70
results = {}

for state, (cx, cy) in states_coords.items():
    x1 = max(cx - crop_w // 2, 0)
    y1 = max(cy - crop_h // 2, 0)
    x2 = min(x1 + crop_w, image.shape[1])
    y2 = min(y1 + crop_h, image.shape[0])
    
    roi_bgr = image[y1:y2, x1:x2]
    
    # Upscale the small ROI to help Tesseract
    roi_bgr = cv2.resize(roi_bgr, None, fx=2.0, fy=2.0, interpolation=cv2.INTER_CUBIC)
    
    # Convert to HSV and isolate dark text
    roi_hsv = cv2.cvtColor(roi_bgr, cv2.COLOR_BGR2HSV)
    ## H: Hue(0-180), S: Saturation(0-255), V: Value for Brightness()
    lower_dark = np.array([0, 0, 0])       # guess for dark
    upper_dark = np.array([180, 255, 80])  # guess for lightness upper bound
    mask = cv2.inRange(roi_hsv, lower_dark, upper_dark) #--inRange()- Apply global threshold
    mask_inv = 255 - mask
    

    # Morphological opening to remove specs
    kernel = np.ones((2,2), np.uint8)
    processed = cv2.morphologyEx(mask_inv, cv2.MORPH_OPEN, kernel)
    #processed = cv2.dilate(processed, kernel, iterations=1)
    #processed = cv2.morphologyEx(processed, cv2.MORPH_CLOSE, kernel)
    
    # OCR - digit only
    text = pytesseract.image_to_string(processed, config="--psm 7 -c tessedit_char_whitelist=0123456789")
    text = text.strip()
    
    # Filter out non-digit characters
    digits_only = re.sub(r"[^0-9]", "", text)
    
    results[state] = digits_only if digits_only else text

for st, val in results.items():
    print(f"{st}: {val}")

Alabama: 135
Arizona: 182
Arkansas: 262
California: 450
Colorado: 457
Connecticut: 40
Delaware: 688
Florida: 206
Georgia: 179
Idaho: 201
Illinois: 314
Indiana: 2
Iowa: 199
Kansas: 237
Kentucky: 293
Louisiana: 339
Maine: 045
Maryland: 688
Massachusetts: 
Michigan: 990
Minnesota: 339
Mississippi: 216
Missouri: 246
Montana: 375
Nebraska: 320
Nevada: 201
New Hampshire: 381
New Jersey: 291
New Mexico: 446
New York: 396
North Carolina: 315
North Dakota: 223
Ohio: 326
Oklahoma: 426
Oregon: 677
Pennsylvania: 2
Rhode Island: 400
South Carolina: 216
South Dakota: 221
Tennessee: 188
Texas: 157
Utah: 386
Vermont: 043
Virginia: 243
Washington: 499
West Virginia: 
Wisconsin: 201
Wyoming: 384


In [39]:
import cv2
import pytesseract
import numpy as np
import re

# If Tesseract is not in your PATH, uncomment and specify its location:
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

# Hard-coded dictionary of approximate center coordinates for each state:
states_coords = {
    "Alabama": (1480, 890),
    "Arizona": (560, 830),
    "Arkansas": (1270, 800),
    "California": (250, 695),
    "Colorado": (780, 605),
    "Connecticut": (2000, 485),
    "Delaware": (1850, 615),
    "Florida": (1660, 1080),
    "Georgia": (1615, 920),
    "Idaho": (450, 360),
    "Illinois": (1380, 550),
    "Indiana": (1490, 590),
    "Iowa": (1230, 460),
    "Kansas": (1040, 640),
    "Kentucky": (1560, 670),
    "Louisiana": (1280, 1000),
    "Maine": (2140, 310),
    "Maryland": (1850, 620),
    "Massachusetts": (1995, 440),
    "Michigan": (1560, 420),
    "Minnesota": (1200, 230),
    "Mississippi": (1380, 880),
    "Missouri": (1270, 630),
    "Montana": (630, 195),
    "Nebraska": (1000, 480),
    "Nevada": (360, 590),
    "New Hampshire": (2060, 390),
    "New Jersey": (1930, 550),
    "New Mexico": (760, 820),
    "New York": (1920, 410),
    "North Carolina": (1760, 760),
    "North Dakota": (990, 170),
    "Ohio": (1630, 545),
    "Oklahoma": (1080, 770),
    "Oregon": (225, 360),
    "Pennsylvania": (1800, 540),
    "Rhode Island": (2060, 485),
    "South Carolina": (1700, 850),
    "South Dakota": (980, 340),
    "Tennessee": (1510, 760),
    "Texas": (1000, 930),
    "Utah": (560, 590),
    "Vermont": (1980, 315),
    "Virginia": (1800, 680),
    "Washington": (235, 178),
    "West Virginia": (1710, 650),
    "Wisconsin": (1360, 330),
    "Wyoming": (715, 410),
}

# Path to your image (2256x1270):
image_path = "../data/img/mental_health.jpg"

# Read the image
image = cv2.imread(image_path)

if image is None:
    raise ValueError("Could not load image. Check the file path.")

crop_w, crop_h = 140, 70
results = {}
state_colors = {}

for state, (cx, cy) in states_coords.items():
    x1 = max(cx - crop_w // 2, 0)
    y1 = max(cy - crop_h // 2, 0)
    x2 = min(x1 + crop_w, image.shape[1] - 1)
    y2 = min(y1 + crop_h, image.shape[0] - 1)
    
    roi_bgr = image[y1:y2, x1:x2]
    
    # Upscale the small ROI to help Tesseract
    roi_bgr = cv2.resize(roi_bgr, None, fx=2.0, fy=2.0, interpolation=cv2.INTER_CUBIC)
    
    # Convert to HSV and isolate dark text
    roi_hsv = cv2.cvtColor(roi_bgr, cv2.COLOR_BGR2HSV)
    lower_dark = np.array([0, 0, 0])       # guess for dark
    upper_dark = np.array([180, 255, 80])  # guess for lightness upper bound
    mask = cv2.inRange(roi_hsv, lower_dark, upper_dark) 
    mask_inv = 255 - mask

    # Morphological opening to remove specs
    kernel = np.ones((2,2), np.uint8)
    processed = cv2.morphologyEx(mask_inv, cv2.MORPH_OPEN, kernel)
    
    # OCR - digit only
    text = pytesseract.image_to_string(processed, config="--psm 7 -c tessedit_char_whitelist=0123456789")
    text = text.strip()
    
    # Filter out non-digit characters
    digits_only = re.sub(r"[^0-9]", "", text)
    
    results[state] = max(min(int(digits_only),677),135) if digits_only else 179

    ### **Extract Non-White, Non-Black State Color** ###
    roi_pixels = roi_bgr.reshape(-1, 3)  # Flatten the ROI to a list of pixels

    # Define color thresholds (tunable)
    min_black = 55   # Below this is black
    max_white = 225  # Above this is white

    # Filter out black and white pixels
    valid_pixels = [
        (r, g, b) for b, g, r in roi_pixels
        if (r > min_black or g > min_black or b > min_black)  # Not black
        and (r < max_white or g < max_white or b < max_white)  # Not white
    ]

    # Compute the average color if valid pixels exist
    if valid_pixels:
        avg_color = np.mean(valid_pixels, axis=0).astype(int)  # Compute mean and convert to int
        state_colors[state] = tuple(avg_color)  # Store as (R, G, B)
    else:
        state_colors[state] = (0, 0, 0)  # Default if no valid color found

# Print extracted numbers and colors
print("\n--- Extracted Numbers ---")
for st, val in results.items():
    print(f"{st}: {val}")

print("\n--- Extracted State Colors (Filtered) ---")
for st, color in state_colors.items():
    print(f"{st}: RGB {color}")

temp_df1 = pd.DataFrame({"state":results.keys(), "mental_health_provider":results.values()})
temp_df2 = pd.DataFrame({"state":state_colors.keys(), "mental_health_RGB":state_colors.values()})
mental_health_df = pd.merge(temp_df1, temp_df2, on='state',how='left')


--- Extracted Numbers ---
Alabama: 135
Arizona: 182
Arkansas: 262
California: 450
Colorado: 457
Connecticut: 135
Delaware: 677
Florida: 206
Georgia: 179
Idaho: 201
Illinois: 314
Indiana: 135
Iowa: 199
Kansas: 237
Kentucky: 293
Louisiana: 339
Maine: 135
Maryland: 677
Massachusetts: 179
Michigan: 677
Minnesota: 339
Mississippi: 216
Missouri: 246
Montana: 375
Nebraska: 320
Nevada: 201
New Hampshire: 381
New Jersey: 291
New Mexico: 446
New York: 396
North Carolina: 315
North Dakota: 223
Ohio: 326
Oklahoma: 426
Oregon: 677
Pennsylvania: 135
Rhode Island: 400
South Carolina: 216
South Dakota: 221
Tennessee: 188
Texas: 157
Utah: 386
Vermont: 135
Virginia: 243
Washington: 499
West Virginia: 179
Wisconsin: 201
Wyoming: 384

--- Extracted State Colors (Filtered) ---
Alabama: RGB (197, 54, 119)
Arizona: RGB (205, 100, 138)
Arkansas: RGB (210, 190, 185)
California: RGB (143, 196, 194)
Colorado: RGB (139, 197, 196)
Connecticut: RGB (162, 197, 194)
Delaware: RGB (191, 179, 180)
Florida: RGB (193, 1

In [47]:
temp_df1 = pd.DataFrame({"state":results.keys(), "mental_health_provider":results.values()})
temp_df2 = pd.DataFrame({"state":state_colors.keys(), "mental_health_RGB":state_colors.values()})
mental_health_df = pd.merge(temp_df1, temp_df2, on='state',how='left')
add_alaska = pd.DataFrame({'state':'Alaska', 'mental_health_provider':723, 'mental_health_RGB':[(97, 186, 185)]})
add_hawaii = pd.DataFrame({'state':'Hawaii', 'mental_health_provider':299, 'mental_health_RGB':[(194, 205, 198)]})

mental_health_df = pd.concat([add_alaska, add_hawaii, mental_health_df], axis=0, ignore_index=True).sort_values('state').reset_index(drop=True)
mental_health_df

,state,mental_health_provider,mental_health_RGB
0,Alabama,135,"(197, 54, 119)"
1,Alaska,723,"(97, 186, 185)"
2,Arizona,182,"(205, 100, 138)"
3,Arkansas,262,"(210, 190, 185)"
4,California,450,"(143, 196, 194)"
5,Colorado,457,"(139, 197, 196)"
6,Connecticut,135,"(162, 197, 194)"
7,Delaware,677,"(191, 179, 180)"
8,Florida,206,"(193, 143, 150)"
9,Georgia,179,"(204, 95, 135)"


In [54]:
import numpy as np
import pandas as pd
import cv2

def get_pipeline_coordinates():
    # Load the image
    image_path = "../data/img/gas_pipe.png"
    image = cv2.imread(image_path)

    # Convert image to RGB
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Define color ranges for blue and orange in RGB format
    blue_lower = np.array([0, 0, 128])  # Approximate lower bound for blue
    blue_upper = np.array([100, 150, 255])  # Approximate upper bound for blue

    orange_lower = np.array([200, 100, 0])  # Approximate lower bound for orange
    orange_upper = np.array([255, 180, 80])  # Approximate upper bound for orange

    # Create masks for blue and orange pixels
    blue_mask = cv2.inRange(image_rgb, blue_lower, blue_upper)
    orange_mask = cv2.inRange(image_rgb, orange_lower, orange_upper)

    # Get pixel coordinates
    blue_pixels = np.column_stack(np.where(blue_mask > 0))
    orange_pixels = np.column_stack(np.where(orange_mask > 0))

    # Mapping function for latitude and longitude
    def pixel_to_geo(x, y, width=901, height=461):
        lat = 25 + (50 - 25) * (1 - y / 460)  # Linear interpolation for latitude
        lon = -125 + (-70 + 125) * (x / 854)  # Linear interpolation for longitude
        return lat, lon

    # Convert pixel coordinates to geo-coordinates
    blue_coords = [pixel_to_geo(x, y) for y, x in blue_pixels]
    orange_coords = [pixel_to_geo(x, y) for y, x in orange_pixels]

    # Create DataFrames for visualization
    blue_df = pd.DataFrame(blue_coords, columns=["latitude", "longitude"])
    orange_df = pd.DataFrame(orange_coords, columns=["latitude", "longitude"])

    return blue_df, orange_df 


def find_nearby_points_fast(haunted_df, blue_df, orange_df, radius_miles=10):
    """
    Quickly finds if any blue or orange points exist within a rough bounding box of ±5 miles.

    Parameters:
    - haunted_df: DataFrame containing haunted place locations with "Latitude" and "Longitude".
    - blue_df: DataFrame containing blue point coordinates with "Latitude" and "Longitude".
    - orange_df: DataFrame containing orange point coordinates with "Latitude" and "Longitude".
    - radius_miles: Distance threshold in miles (default=10, meaning ±5 miles).

    Returns:
    - A DataFrame indicating if there are nearby blue or orange points for each haunted place.
    """

    # Convert miles to latitude degrees (1 mile ≈ 0.0145 degrees)
    mile_to_degree = 0.0145  # Approximate conversion factor

    results = []
    
    for _, hplace in haunted_df.iterrows():
        hplace_state, hplace_lat, hplace_lon = hplace['state'], hplace["latitude"], hplace["longitude"]

        # Compute bounding box (±5 miles)
        lat_range = radius_miles / 2 * mile_to_degree
        lon_range = (radius_miles / 2 * mile_to_degree) / np.cos(np.radians(hplace_lat))

        lat_min, lat_max = hplace_lat - lat_range, hplace_lat + lat_range
        lon_min, lon_max = hplace_lon - lon_range, hplace_lon + lon_range

        # Filter blue and orange points inside the bounding box
        blue_nearby = not blue_df[
            (blue_df["latitude"] >= lat_min) & (blue_df["latitude"] <= lat_max) &
            (blue_df["longitude"] >= lon_min) & (blue_df["longitude"] <= lon_max)
        ].empty

        orange_nearby = not orange_df[
            (orange_df["latitude"] >= lat_min) & (orange_df["latitude"] <= lat_max) &
            (orange_df["longitude"] >= lon_min) & (orange_df["longitude"] <= lon_max)
        ].empty

        results.append({
            "state": hplace_state,
            "latitude": hplace_lat,
            "longitude": hplace_lon,
            "intrastate_gaspipe_within_10miles": blue_nearby,
            "interstate_gaspipe_within_10miles": orange_nearby
        })
    
    return  pd.DataFrame(results)

# Example usage:
blue_df, orange_df = get_pipeline_coordinates()
haunted_df = pd.read_csv('../data/haunted_places.tsv',sep='\t')
result_df = find_nearby_points_fast(haunted_df, blue_df, orange_df)
# print(result_df)


In [55]:
result_df = result_df.merge(mental_health_df, how='left', on='state')
result_df.to_csv('../data/arcgis_img.csv')
result_df

,state,latitude,longitude,intrastate_gaspipe_within_10miles,interstate_gaspipe_within_10miles,mental_health_provider,mental_health_RGB
0,Michigan,42.962106,-85.504893,False,False,677.0,"(187, 204, 199)"
1,Michigan,41.971425,-84.381843,False,False,677.0,"(187, 204, 199)"
2,Michigan,41.904538,-84.035656,False,False,677.0,"(187, 204, 199)"
3,Michigan,41.905712,-84.017565,False,False,677.0,"(187, 204, 199)"
4,Michigan,42.244006,-84.745177,True,False,677.0,"(187, 204, 199)"
...,...,...,...,...,...,...,...
10987,Colorado,39.862610,-105.048936,False,False,457.0,"(139, 197, 196)"
10988,Colorado,39.847237,-105.032091,False,False,457.0,"(139, 197, 196)"
10989,Colorado,39.769726,-105.063974,False,False,457.0,"(139, 197, 196)"
10990,Colorado,39.764055,-105.103613,False,False,457.0,"(139, 197, 196)"


In [6]:
import os 
def file_chekcer(keyword:str, file_type:str):
    """
    File checker inspect a file of given name is available and, if yes, return it. 
    This function is used in data_join() function. 
    """
    # check raw data is prepared for joining 
    found_csvs = []
    for filename in os.listdir('../data'):
        # We check that it's a CSV file (filename ends with .tsv)
        # and that 'keyword' appears in the filename
        if filename.endswith(file_type) and keyword in filename:
            found_csvs.append(filename)
    return found_csvs
 

print(f"../data/{file_chekcer('daylight','tsv')[0]}" )

../data/daylight_added.tsv


In [2]:
import shutil
# Find Tesseract path automatically
tesseract_path = shutil.which("tesseract")
print(tesseract_path)

# If not found, set it manually
if not tesseract_path:
    import platform
    os_name = platform.system()
    if os_name == "Windows":
        print("windows!!!")
        tesseract_path = r'C:\Program Files\Tesseract-OCR\tesseract.exe'  # Change if installed elsewhere
    elif os_name == "Darwin":  # macOS
        tesseract_path = "/opt/homebrew/bin/tesseract"
    elif os_name == "Linux":
        tesseract_path = "/usr/bin/tesseract"  # Adjust based on your installation
    else:
        raise FileNotFoundError("Tesseract-OCR not found. ")

tesseract_path

None
windows!!!


'C:\\Program Files\\Tesseract-OCR\\tesseract.exe'

In [25]:
daylight_df = pd.read_csv('../data/daylight_added.tsv',sep='\t') 
merged_df = pd.read_csv('../data/haunted_places.tsv',sep='\t')
print(merged_df.shape)

daylight_df = daylight_df[['daylight_minutes']]
print(daylight_df.shape)

#merged_df = pd.merge(merged_df, daylight_df, on=('description','city_latitude','city_longitude'), how='left')
merged_df = pd.concat([merged_df, daylight_df], axis=1)
print(merged_df.shape) 
merged_df
#merged_df[merged_df.duplicated(subset=['description','city_latitude','city_longitude'],keep=False)]

(10992, 11)
(10992, 1)
(10992, 12)


,city,country,description,location,state,state_abbrev,longitude,latitude,city_longitude,city_latitude,date_occured,daylight_minutes
0,Ada,United States,Ada witch - Sometimes you can see a misty blue...,Ada Cemetery,Michigan,MI,-85.504893,42.962106,-85.495480,42.960727,2025-02-20,393.0
1,Addison,United States,A little girl was killed suddenly while waitin...,North Adams Rd.,Michigan,MI,-84.381843,41.971425,-84.347168,41.986434,2025-01-01,676.0
2,Adrian,United States,If you take Gorman Rd. west towards Sand Creek...,Ghost Trestle,Michigan,MI,-84.035656,41.904538,-84.037166,41.897547,2025-01-01,552.0
3,Adrian,United States,"In the 1970's, one room, room 211, in the old ...",Siena Heights University,Michigan,MI,-84.017565,41.905712,-84.037166,41.897547,1970-02-23,382.0
4,Albion,United States,Kappa Delta Sorority - The Kappa Delta Sororit...,Albion College,Michigan,MI,-84.745177,42.244006,-84.753030,42.243097,2025-01-01,550.0
...,...,...,...,...,...,...,...,...,...,...,...,...
10987,Westminster,United States,at 12 midnight you can see a lady with two lit...,city hall,Colorado,CO,-105.048936,39.862610,-105.037205,39.836653,2025-01-01,313.0
10988,Westminster,United States,Is haunted by the victims of a murder that hap...,Pillar of Fire,Colorado,CO,-105.032091,39.847237,-105.037205,39.836653,2025-01-01,313.0
10989,Wheat Ridge,United States,The institution was for kids 18 years old and ...,Ridge Mental Institution,Colorado,CO,-105.063974,39.769726,-105.077206,39.766098,2025-02-18,689.0
10990,Wheat Ridge,United States,Gymnasium - their have been reports of a litt...,Wheat Ridge Middle School,Colorado,CO,-105.103613,39.764055,-105.077206,39.766098,2025-01-01,313.0
